# Step 1: a SCEC CFM fault surface into Python



### The example fault

We use the **San Bernardino Mountains section of the San Andreas fault**, CFM 6.1
preferred model, at the ~500 m semi-regularised resolution:

```
SAFS-SAFZ-SBMT-San_Andreas_fault-CFM6_500m.ts
```

CFM's object names are hierarchical, which is worth decoding once:

| Token | Meaning |
|---|---|
| `SAFS` | San Andreas **fault system** |
| `SAFZ` | San Andreas **fault zone** |
| `SBMT` | San Bernardino Mountains section |
| `San_Andreas_fault` | fault name |
| `CFM6` | the version in which this *representation* was introduced |
| `500m` | mesh resolution directory |

Note that `CFM6` labels the fault representation, not the model release — a CFM 6.1
archive contains objects labelled `CFM4`, `CFM5`, `CFM6`, and so on.

### Data provenance

- **Source:** SCEC Community Fault Model, version 6.1
- **DOI:** [10.5281/zenodo.8327463](https://doi.org/10.5281/zenodo.8327463)
- **License:** BSD 3-Clause
- **Citation:** Plesch, A., et al. (2007), *Community Fault Model (CFM) for Southern
  California*, BSSA 97, 1793-1802, plus the Zenodo DOI above.

We deliberately pin the **versioned** record (`8327463`) rather than the concept DOI, so this notebook keeps returning the same geometry after CFM 7 and later releases.

## 1. Setup

`remotezip` is optional but strongly recommended: it uses HTTP range requests to pull
only the one t-surf file we need (a few hundred kB) instead of the whole 60 MB archive.

In [2]:
import hashlib
import pathlib

import numpy as np
import plotly.graph_objects as go
import requests

In [3]:
# --- CFM 6.1, pinned to the Zenodo versioned record -------------------------
ZENODO_RECORD = "8327463"
ZIP_NAME      = "CFM6.1_release_2023.zip"
ZIP_URL       = f"https://zenodo.org/records/{ZENODO_RECORD}/files/{ZIP_NAME}?download=1"
ZIP_MD5       = "5d28c738b6fa1eaf2ce491e5284bb729"

# The fault object we want, and the resolution directory it lives in.
FAULT   = "SAFS-SAFZ-SBMT-San_Andreas_fault-CFM6_500m.ts"
RES_DIR = "/preferred/500m/"

CACHE = pathlib.Path("cfm_cache")
CACHE.mkdir(exist_ok=True)

## 2. Fetch the fault

Two paths. The fast one reads the zip's central directory over HTTP and extracts a
single member. The fallback downloads the whole archive once and caches it, checking
the md5 so you know you have exactly the released bytes.

The CFM archive has a top-level directory inside the zip whose name we do not want to
hard-code, so we match on the path *suffix* instead of the full path.

In [ ]:
def fetch_tsurf(fault=FAULT, res_dir=RES_DIR):
    """Return the text of one CFM t-surf file, caching it locally."""
    local = CACHE / fault
    if local.exists():
        print(f"cached: {local}")
        return local.read_text()

    # --- fast path: range requests, pull one member only --------------------
    try:
        from remotezip import RemoteZip
        with RemoteZip(ZIP_URL) as z:
            names = [n for n in z.namelist() if res_dir in n and n.endswith(fault)]
            if not names:
                raise FileNotFoundError(f"{fault} not found under *{res_dir}")
            print(f"extracting (range request): {names[0]}")
            text = z.read(names[0]).decode("utf-8", errors="replace")
            local.write_text(text)
            return text
    except Exception as exc:
        print(f"range-request path unavailable ({exc!r}); downloading full archive")

    # --- fallback: whole archive, cached and checksummed --------------------
    import zipfile

    archive = CACHE / ZIP_NAME
    if not archive.exists():
        with requests.get(ZIP_URL, stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(archive, "wb") as fh:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    fh.write(chunk)

    digest = hashlib.md5(archive.read_bytes()).hexdigest()
    print(f"md5 {digest} ({'ok' if digest == ZIP_MD5 else 'MISMATCH'})")

    with zipfile.ZipFile(archive) as z:
        names = [n for n in z.namelist() if res_dir in n and n.endswith(fault)]
        if not names:
            raise FileNotFoundError(f"{fault} not found under *{res_dir}")
        text = z.read(names[0]).decode("utf-8", errors="replace")

    local.write_text(text)
    return text


raw = fetch_tsurf()
print(f"\n{len(raw.splitlines())} lines\n")
print("\n".join(raw.splitlines()[:20]))

## 3. Parsing GOCAD t-surf

The format is plain text and only a handful of keywords matter:

| Keyword | Meaning |
|---|---|
| `TFACE` | starts a new triangulated patch; one file may hold several |
| `VRTX id x y z` | a vertex |
| `PVRTX id x y z p1 p2 ...` | a vertex carrying property values |
| `ATOM id ref_id` | a *new* vertex id that reuses the position of `ref_id` |
| `TRGL i j k` | a triangle, referring to vertex **ids** |
| `BSTONE` / `BORDER` | border-stone bookkeeping — safely ignored here |

Three things catch people out:

1. **`ATOM` lines.** They exist so patches can share vertices. Skip them and your
   index map silently loses entries, and `TRGL` lookups start failing.
2. **Vertex ids are 1-based and not necessarily contiguous**, and in multi-patch files
   they continue across `TFACE` boundaries. Build an `id -> row` map; never assume
   ordering.
3. **`ZPOSITIVE`** in the header declares whether z is elevation or depth. CFM uses
   elevation (negative below sea level), but we read the header and report it rather
   than assume.

Coordinates are metres, UTM zone 11 / NAD27 (EPSG:26711).

The parser, `read_tsurf`, lives in
[`local_py_scripts/notebook_utils.py`](../../local_py_scripts/notebook_utils.py) so
every notebook that reads a CFM `.ts` file shares one implementation.

In [ ]:
import sys

sys.path.append(str(pathlib.Path("../../../local_py_scripts").resolve()))
from notebook_utils import read_tsurf

patches, header = read_tsurf(raw)

print(f"header      : {header}")
print(f"patches     : {len(patches)}")
for i, p in enumerate(patches):
    print(f"  patch {i}: {len(p['points']):6d} vertices, {len(p['cells']):6d} triangles")

## 4. Sanity checks before doing anything else

k223d computes distances by trilateration across the mesh, so the topology has to be
sound: a single connected sheet, manifold edges, and a well-defined outer boundary.
Cheap checks now are much cheaper than confusing results later.

- **Edge multiplicity.** Every interior edge is shared by exactly 2 triangles; boundary
  edges by 1. Anything shared by 3 or more is non-manifold and will need fixing.
- **Euler characteristic.** For a single triangulated disc, `V - E + F = 1`. Getting
  0 suggests a handle or a seam; getting 2 suggests a closed surface.
- **Connectivity.** Number of connected components should be 1.

`mesh_report`, used below, also lives in `notebook_utils.py` — the same function runs
this check on every mesh in this series, before and after remeshing.

In [ ]:
from notebook_utils import mesh_report

points = patches[0]["points"]
cells  = patches[0]["cells"]
areas  = mesh_report(points, cells, "CFM 6.1 native 500m mesh")

The `z` range tells us immediately whether this fault daylights. If `z.max()` is at or
near 0 the surface reaches the free surface, and we will want k223d's surface-rupture
handling in a later step; if it is well below 0 the fault is blind and we can ignore
that machinery entirely.

The spread between the minimum and maximum triangle edge scale is the number that
matters for step 2. A semi-regularised CFM mesh is far better behaved than the native
one, but it is still not what you would choose for a geodesic-distance solver — hence
the gmsh remesh.

## 5. Plotting

We use plotly rather than pyvista here so the notebook renders on GitHub, nbviewer,
Colab and Binder without a live VTK/trame server behind it. `go.Mesh3d` takes exactly
the arrays we already have: `x, y, z` plus `i, j, k` triangle indices.

Two details worth copying into your own code:

- `aspectmode="data"` is essential. Without it plotly normalises each axis
  independently and an 80 km x 20 km fault gets drawn as a cube, making the dip
  meaningless.
- Coordinates are shifted to a local origin and converted to km **for display only**.
  The full-precision UTM arrays stay untouched for the rest of the pipeline.

The same helper is reused in later notebooks with `scalars` set to slip (per cell) or
rupture time (per vertex), which is why it takes an `intensitymode`.

In [ ]:
def plot_mesh(points, cells, scalars=None, intensitymode="cell",
              title="", cbar_title="", wireframe=True, origin=None):
    """Interactive 3D plot of a triangulated fault surface.

    scalars       : per-cell or per-vertex values (optional)
    intensitymode : "cell" for cell data (k223d slip), "vertex" for nodal data
                    (k223d rupture time)
    origin        : (x0, y0) subtracted before scaling to km; defaults to the
                    mesh centroid. Pass the same origin to compare two meshes.
    """
    if origin is None:
        origin = points[:, :2].mean(axis=0)
    x = (points[:, 0] - origin[0]) / 1e3
    y = (points[:, 1] - origin[1]) / 1e3
    z = points[:, 2] / 1e3

    mesh_kw = dict(x=x, y=y, z=z,
                   i=cells[:, 0], j=cells[:, 1], k=cells[:, 2],
                   flatshading=True, name="")
    if scalars is None:
        mesh_kw.update(color="#8fa8c8", opacity=1.0)
    else:
        mesh_kw.update(intensity=np.asarray(scalars),
                       intensitymode=intensitymode,
                       colorscale="Viridis",
                       colorbar=dict(title=cbar_title, len=0.6))

    traces = [go.Mesh3d(**mesh_kw)]

    if wireframe:
        # unique edges, drawn as one Scatter3d trace with None separators
        seen = set()
        ex, ey, ez = [], [], []
        for a, b, c in cells:
            for u, v in ((a, b), (b, c), (c, a)):
                e = (min(u, v), max(u, v))
                if e in seen:
                    continue
                seen.add(e)
                ex += [x[u], x[v], None]
                ey += [y[u], y[v], None]
                ez += [z[u], z[v], None]
        traces.append(go.Scatter3d(x=ex, y=ey, z=ez, mode="lines",
                                   line=dict(color="rgba(20,20,20,0.35)", width=1),
                                   hoverinfo="skip", showlegend=False))

    fig = go.Figure(traces)
    fig.update_layout(
        title=title,
        margin=dict(l=0, r=0, t=40, b=0),
        height=600,
        scene=dict(aspectmode="data",
                   xaxis_title="east (km)",
                   yaxis_title="north (km)",
                   zaxis_title="elevation (km)"),
    )
    return fig, origin


fig, ORIGIN = plot_mesh(
    points, cells,
    title=f"{FAULT}  —  CFM 6.1, ~500 m semi-regularised mesh",
)
fig.show()

Colouring by triangle size makes the mesh irregularity obvious, and gives a baseline to
compare against after the gmsh remesh in step 2.

In [ ]:
fig, _ = plot_mesh(
    points, cells,
    scalars=np.sqrt(areas),
    intensitymode="cell",
    title="Triangle edge scale of the CFM mesh",
    cbar_title="sqrt(area)<br>(m)",
    origin=ORIGIN,
)
fig.show()

## 6. Save for step 2

We write an STL, which is what gmsh wants as the input to its surface reparametrisation
(`classifySurfaces` / `createGeometry`). STL is vertex-duplicating and carries no
connectivity, but that is fine — gmsh rebuilds the topology itself, and we have already
verified above that the topology is sound.

Coordinates stay in metres, UTM zone 11 / NAD27. Reprojection to a local frame is a
question for later; for a fault this close to the centre of zone 11 the distortion is
small, but it is worth revisiting before quoting absolute rupture areas.

As an alternative, `write_vtk_fields` from [`local_py_scripts/notebook_utils.py`](../../local_py_scripts/notebook_utils.py)
writes a legacy ASCII VTK (`POLYDATA`) file instead. gmsh reads `.vtk` just as
happily as `.stl`, and unlike STL it keeps explicit connectivity (no vertex
duplication) and can carry cell/point fields — handy in later steps once we have
slip or rupture time to attach to the mesh.

In [ ]:
# def write_stl(path, points, cells, name="fault"):
#     """Minimal ASCII STL writer."""
#     p = points[cells]
#     n = np.cross(p[:, 1] - p[:, 0], p[:, 2] - p[:, 0])
#     n /= np.linalg.norm(n, axis=1)[:, None]

#     with open(path, "w") as fh:
#         fh.write(f"solid {name}\n")
#         for tri, nrm in zip(p, n):
#             fh.write(f"  facet normal {nrm[0]:.6e} {nrm[1]:.6e} {nrm[2]:.6e}\n")
#             fh.write("    outer loop\n")
#             for v in tri:
#                 fh.write(f"      vertex {v[0]:.6e} {v[1]:.6e} {v[2]:.6e}\n")
#             fh.write("    endloop\n  endfacet\n")
#         fh.write(f"endsolid {name}\n")


# stl_path = CACHE / "SBMT_San_Andreas_CFM6_500m.stl"
# write_stl(stl_path, points, cells, name="SBMT_San_Andreas")
# print(f"wrote {stl_path}  ({stl_path.stat().st_size / 1e6:.1f} MB)")

In [ ]:
from notebook_utils import write_vtk_fields

vtk_path = CACHE / "SBMT_San_Andreas_CFM6_500m.vtk"
write_vtk_fields(vtk_path, points, cells)
print(f"wrote {vtk_path}  ({vtk_path.stat().st_size / 1e6:.1f} MB)")

## What we have

- One CFM fault surface, fetched reproducibly from a pinned Zenodo record.
- A GOCAD t-surf parser that handles `ATOM` aliases, non-contiguous vertex ids and
  multi-`TFACE` files.
- Topology and geometry checks confirming a single manifold sheet.
- A plotting helper that we will reuse unchanged for k223d's cell-centred slip and
  node-centred rupture times.
- An STL ready for gmsh.

**Next:** step 2 remeshes this surface to a target element size, which is what k223d
actually wants — the distance algorithm behaves far better on a quasi-uniform
triangulation than on the variable-resolution mesh CFM ships.